# Feature Engineering & Toxicity Labelling

## Objective
Build the feature matrix that feeds the toxicity classifier. Each row is one trade, 
each column is a feature describing market conditions at the moment of that trade, 
plus a binary label indicating whether the trade was toxic.

## Features — Two Sources

**Book-derived features** (from order book reconstruction):
- Spread, microprice, midprice
- Depth imbalance at levels 1, 5, 10, 25
- Bid/ask pressure concentration

**Trade-derived features** (from trade data alone):
- Trade intensity (rolling trade count)
- Volume acceleration
- Signed volume imbalance

## Toxicity Definition
A trade is toxic if the price moves adversely by >8 bps within 10 seconds, 
in the direction of the trade. Threshold chosen from the 90th percentile of 
absolute 10-second forward returns. ~5.5% of trades are labelled toxic.

## Key Finding
[Leave this blank — fill it in after you've computed everything and noticed patterns]

In [ ]:
import pandas as pd
import numpy as np

trades = pd.read_parquet("../data/processed/BTCUSDT/week1.parquet")

# filter to Sep 9
ts_ms = trades["timestamp"].astype("int64") // 10**6
day_start = pd.Timestamp("2024-09-09").value // 10**6
day_end = pd.Timestamp("2024-09-10").value // 10**6
trades = trades[(ts_ms >= day_start) & (ts_ms < day_end)].copy()

print(trades.columns.tolist())
print(trades.head(3))
print(len(trades))

['timestamp', 'price', 'qty', 'sign']
                      timestamp    price    qty  sign
0 2024-09-09 00:00:00.399800062  54849.9  0.001    -1
1 2024-09-09 00:00:00.758599997  54850.0  0.002     1
2 2024-09-09 00:00:00.835400105  54850.0  0.001     1
2140818


In [ ]:
# convert timestamp to milliseconds for fast lookup
trades["ts_ms"] = trades["timestamp"].astype("int64") // 10**6
prices = trades["price"].values
ts = trades["ts_ms"].values

# for each horizon, find the price at t + horizon and compute return in bps
horizons_sec = [1, 5, 10, 30]

for h in horizons_sec:
    h_ms = h * 1000
    # for each trade, find the index of the first trade at least h seconds later
    future_idx = np.searchsorted(ts, ts + h_ms)
    # clip to valid range
    future_idx = np.clip(future_idx, 0, len(prices) - 1)
    # forward return in bps
    fwd_return = (prices[future_idx] - prices) / prices * 10000
    trades[f"fwd_{h}s_bps"] = fwd_return

# now look at the distributions
for h in horizons_sec:
    col = f"fwd_{h}s_bps"
    abs_col = trades[col].abs()
    print(f"\n--- {h}s forward return (absolute, bps) ---")
    print(f"  mean:   {abs_col.mean():.2f}")
    print(f"  median: {abs_col.median():.2f}")
    print(f"  75th:   {abs_col.quantile(0.75):.2f}")
    print(f"  90th:   {abs_col.quantile(0.90):.2f}")
    print(f"  95th:   {abs_col.quantile(0.95):.2f}")
    print(f"  99th:   {abs_col.quantile(0.99):.2f}")


--- 1s forward return (absolute, bps) ---
  mean:   1.70
  median: 0.89
  75th:   1.93
  90th:   3.57
  95th:   5.42
  99th:   17.11

--- 5s forward return (absolute, bps) ---
  mean:   3.05
  median: 1.75
  75th:   3.54
  90th:   6.19
  95th:   8.90
  99th:   28.32

--- 10s forward return (absolute, bps) ---
  mean:   3.84
  median: 2.42
  75th:   4.79
  90th:   8.09
  95th:   11.11
  99th:   30.50

--- 30s forward return (absolute, bps) ---
  mean:   6.19
  median: 4.08
  75th:   7.85
  90th:   13.18
  95th:   18.11
  99th:   36.01


In [4]:
# signed forward return: positive means price went up
fwd_10s = trades["fwd_10s_bps"].values
signs = trades["sign"].values

# toxic = trade direction matches a large forward move
# buy (sign=1) followed by price UP (fwd > 8) = toxic
# sell (sign=-1) followed by price DOWN (fwd < -8) = toxic
trades["toxic"] = ((signs == 1) & (fwd_10s > 8)) | ((signs == -1) & (fwd_10s < -8))

toxic_rate = trades["toxic"].mean()
print(f"Toxic rate: {toxic_rate:.4f} ({toxic_rate*100:.1f}%)")
print(f"Toxic trades: {trades['toxic'].sum()}")
print(f"Total trades: {len(trades)}")

Toxic rate: 0.0553 (5.5%)
Toxic trades: 118305
Total trades: 2140818


In [5]:
toxic_trades = trades[trades["toxic"]].copy()
toxic_ts = toxic_trades["ts_ms"].values

# time gaps between consecutive toxic trades (in seconds)
gaps = np.diff(toxic_ts) / 1000
print(f"Median gap between toxic trades: {np.median(gaps):.1f}s")
print(f"Mean gap between toxic trades: {np.mean(gaps):.1f}s")
print(f"Gaps < 1s: {(gaps < 1).sum()} ({(gaps < 1).mean()*100:.1f}%)")
print(f"Gaps < 5s: {(gaps < 5).sum()} ({(gaps < 5).mean()*100:.1f}%)")

# if toxic trades were uniformly distributed, expected gap would be:
total_duration_s = (ts[-1] - ts[0]) / 1000
expected_gap = total_duration_s / len(toxic_trades)
print(f"\nExpected gap if uniform: {expected_gap:.1f}s")
print(f"Actual median gap: {np.median(gaps):.1f}s")


Median gap between toxic trades: 0.0s
Mean gap between toxic trades: 0.7s
Gaps < 1s: 117367 (99.2%)
Gaps < 5s: 117764 (99.5%)

Expected gap if uniform: 0.7s
Actual median gap: 0.0s


In [7]:
import pandas as pd
import numpy as np

trades = pd.read_parquet("../data/processed/BTCUSDT/week1.parquet")
ts_ms = trades["timestamp"].astype("int64") // 10**6
day_start = pd.Timestamp("2024-09-09").value // 10**6
day_end = pd.Timestamp("2024-09-10").value // 10**6
trades = trades[(ts_ms >= day_start) & (ts_ms < day_end)].copy()
trades["ts_ms"] = trades["timestamp"].astype("int64") // 10**6

In [8]:

# trade intensity: how many trades occurred in the last N seconds?

ts = trades["ts_ms"].values

for window_s in [1, 5, 10]:
    window_ms = window_s * 1000
    lookback_idx = np.searchsorted(ts, ts - window_ms)
    trades[f"trade_intensity_{window_s}s"] = np.arange(len(ts)) - lookback_idx

print(trades[["trade_intensity_1s", "trade_intensity_5s", "trade_intensity_10s"]].describe())

       trade_intensity_1s  trade_intensity_5s  trade_intensity_10s
count        2.140818e+06        2.140818e+06         2.140818e+06
mean         3.558935e+02        8.553905e+02         1.256033e+03
std          9.378906e+02        2.455912e+03         3.187689e+03
min          0.000000e+00        0.000000e+00         0.000000e+00
25%          2.000000e+01        8.800000e+01         1.730000e+02
50%          6.700000e+01        2.140000e+02         3.810000e+02
75%          2.590000e+02        5.780000e+02         9.160000e+02
max          1.041400e+04        2.719200e+04         3.382900e+04


In [ ]:
qty = trades["qty"].values
signs = trades["sign"].values
ts = trades["ts_ms"].values
cum_qty = np.cumsum(qty)
cum_signed_qty = np.cumsum(signs * qty)

# lookback indices (time-based)
idx_5s = np.searchsorted(ts, ts - 5000)
idx_10s = np.searchsorted(ts, ts - 10000)
idx_30s = np.searchsorted(ts, ts - 30000)

# volume in last 5s and last 30s
vol_5s = cum_qty - cum_qty[idx_5s]
vol_30s = cum_qty - cum_qty[idx_30s]

# acceleration: actual 5s volume / expected 5s volume if uniform over 30s
expected_5s = vol_30s * (5 / 30)
trades["volume_acceleration"] = np.where(expected_5s > 0, vol_5s / expected_5s, 1.0)

# signed volume imbalance in last 10s
trades["signed_vol_imbalance_10s"] = cum_signed_qty - cum_signed_qty[idx_10s]

print(trades["volume_acceleration"].describe())
print(trades["signed_vol_imbalance_10s"].describe())


count    2.140818e+06
mean     1.707375e+00
std      1.375938e+00
min      0.000000e+00
25%      6.425007e-01
50%      1.259766e+00
75%      2.449424e+00
max      6.000000e+00
Name: volume_acceleration, dtype: float64
count    2.140818e+06
mean     2.871767e+01
std      1.523067e+02
min     -5.394380e+02
25%     -9.008000e+00
50%      6.440000e-01
75%      1.553900e+01
max      1.399739e+03
Name: signed_vol_imbalance_10s, dtype: float64


/var/folders/d2/s1slzqf17_ld7x84yd68h6740000gn/T/ipykernel_94258/994599994.py:18: RuntimeWarning: invalid value encountered in divide
  trades["volume_acceleration"] = np.where(expected_5s > 0, vol_5s / expected_5s, 1.0)


In [16]:
book_features = pd.read_parquet("../data/processed/features/BTCUSDT_2024-09-09_book_features.parquet")
print(f"Book features shape: {book_features.shape}")
print(f"Trades shape: {trades.shape}")
print(f"Book timestamp range: {book_features['timestamp'].min()} - {book_features['timestamp'].max()}")
print(f"Trade ts_ms range: {trades['ts_ms'].min()} - {trades['ts_ms'].max()}")

Book features shape: (2140813, 11)
Trades shape: (2140818, 10)
Book timestamp range: 1725840001926 - 1725926399930
Trade ts_ms range: 1725840000399 - 1725926399930


In [17]:
# the book features are missing the first 5 trades (before first snapshot)
# find which trades have matching book features
trades_with_books = trades[trades["ts_ms"] >= book_features["timestamp"].iloc[0]].copy()
print(f"Trades after first snapshot: {len(trades_with_books)}")
print(f"Book features: {len(book_features)}")

# should be equal — if so, we can just paste the columns together
if len(trades_with_books) == len(book_features):
    for col in book_features.columns:
        if col != "timestamp":
            trades_with_books[col] = book_features[col].values
    print("Joined successfully")
else:
    print("LENGTH MISMATCH — need to debug")

print(f"Final shape: {trades_with_books.shape}")
print(trades_with_books.columns.tolist())

Trades after first snapshot: 2140813
Book features: 2140813
Joined successfully
Final shape: (2140813, 20)
['timestamp', 'price', 'qty', 'sign', 'ts_ms', 'trade_intensity_1s', 'trade_intensity_5s', 'trade_intensity_10s', 'volume_acceleration', 'signed_vol_imbalance_10s', 'spread', 'microprice', 'midprice', 'depth_imbalance_1', 'depth_imbalance_5', 'depth_imbalance_10', 'depth_imbalance_25', 'bid_pressure', 'ask_pressure', 'pressure_imbalance']


In [18]:
# recompute toxicity label on the trimmed dataframe
prices = trades_with_books["price"].values
ts = trades_with_books["ts_ms"].values
signs = trades_with_books["sign"].values

future_idx = np.searchsorted(ts, ts + 10000)
future_idx = np.clip(future_idx, 0, len(prices) - 1)
fwd_10s_bps = (prices[future_idx] - prices) / prices * 10000

trades_with_books["fwd_10s_bps"] = fwd_10s_bps
trades_with_books["toxic"] = ((signs == 1) & (fwd_10s_bps > 8)) | ((signs == -1) & (fwd_10s_bps < -8))

print(f"Toxic rate: {trades_with_books['toxic'].mean():.4f}")
print(f"Final shape: {trades_with_books.shape}")
print(f"Columns: {trades_with_books.columns.tolist()}")

# save
trades_with_books.to_parquet("../data/processed/features/BTCUSDT_2024-09-09_full_features.parquet", index=False)
print("Saved.")

Toxic rate: 0.0553
Final shape: (2140813, 22)
Columns: ['timestamp', 'price', 'qty', 'sign', 'ts_ms', 'trade_intensity_1s', 'trade_intensity_5s', 'trade_intensity_10s', 'volume_acceleration', 'signed_vol_imbalance_10s', 'spread', 'microprice', 'midprice', 'depth_imbalance_1', 'depth_imbalance_5', 'depth_imbalance_10', 'depth_imbalance_25', 'bid_pressure', 'ask_pressure', 'pressure_imbalance', 'fwd_10s_bps', 'toxic']
Saved.


In [19]:
# compare feature means for toxic vs non-toxic trades
toxic = trades_with_books[trades_with_books["toxic"]]
non_toxic = trades_with_books[~trades_with_books["toxic"]]

feature_cols = ["spread", "depth_imbalance_1", "trade_intensity_1s", 
                "volume_acceleration", "signed_vol_imbalance_10s", "pressure_imbalance"]

for col in feature_cols:
    t_mean = toxic[col].mean()
    nt_mean = non_toxic[col].mean()
    print(f"{col:30s}  toxic={t_mean:8.3f}  non_toxic={nt_mean:8.3f}  ratio={t_mean/nt_mean if nt_mean != 0 else 'inf':.2f}")

spread                          toxic=   0.593  non_toxic=   0.246  ratio=2.41
depth_imbalance_1               toxic=   0.045  non_toxic=   0.012  ratio=3.72
trade_intensity_1s              toxic=1044.061  non_toxic= 315.641  ratio=3.31
volume_acceleration             toxic=   1.979  non_toxic=   1.691  ratio=1.17
signed_vol_imbalance_10s        toxic= 149.999  non_toxic=  21.623  ratio=6.94
pressure_imbalance              toxic=   0.000  non_toxic=   0.006  ratio=0.07
